# Put the bot online — free, in about five minutes

This gives you a **public link anyone can open**, with the fine-tuned model
actually running. No payment, no new accounts.

## Why not Hugging Face?

Hugging Face changed its rules: creating a Gradio Space that runs on compute now
requires a PRO subscription. The free exception is ZeroGPU, limited to accounts
over 30 days old with a verified email, and 5 minutes of GPU time a day. If your
account is new, that door is shut for a month.

Colab plus Gradio's own share tunnel needs neither. The link lives for **one
week** and works from any phone or laptop while this notebook is running.

## How to run it

1. **Runtime → Change runtime type → T4 GPU → Save** (CPU works too; this is a
   77M-parameter model)
2. **Runtime → Run all**, and allow Google Drive access when asked
3. Scroll to the last cell and copy the `https://….gradio.live` link

**Keep this tab open.** The link dies when the notebook stops, so start it before
a demo rather than the night before. For a recording, that is usually what you
want anyway.

In [ ]:
# ----------------------------------------------------------------- settings
REPO_URL = "https://github.com/suryanshu-g/legal-llm-bot.git"

# The model trained in Phase 3.6. Change only if you saved it elsewhere.
MODEL_DIR = "/content/drive/MyDrive/legal-llm-bot/flan-t5-small-context-v3"

In [ ]:
%pip install -q -U "transformers<5" sentence-transformers faiss-cpu "gradio>=4.44,<6"

> **If Colab shows a "RESTART SESSION" button after the install, click it**, then
> carry on from the next cell. Nothing above needs running again.

In [ ]:
import os, subprocess, sys

REPO_DIR = "/content/legal-llm-bot"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--quiet"], check=False)
    print("repository already present, pulled the latest")
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    if r.returncode != 0:
        raise RuntimeError("git clone failed")

sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))

from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(
        f"No model at {MODEL_DIR}.\n"
        "Check the path in the settings cell, or run "
        "notebooks/finetune_flan_t5_small_contextaware.ipynb first.")
print("model:", MODEL_DIR)

---

## Loading

`scripts/bot.py` is the whole assistant: it assembles the context, prompts the
model and reports the passages the answer rests on. `load_model` there also
handles a trap worth knowing about — the checkpoint stores a separate trained
output layer that older `transformers` versions quietly tie away, producing
fluent nonsense with no warning.

In [ ]:
import time
from bot import DISCLAIMER, Bot

t0 = time.time()
bot = Bot(model_dir=MODEL_DIR)
print(f"loaded in {time.time() - t0:.1f}s on {bot.device}")
print(f"{len(bot.retriever.meta)} passages | "
      f"{len(bot.counterparts)} provisions with counterparts")

# A known answer, before anyone else sees it.
probe = bot.ask("Which BNS section replaced IPC Section 302?")
print("\nprobe:", probe.text[:160])
assert "103" in probe.text, "the model is not answering correctly - stop and check it"
print("probe OK")

---

## The interface

The model's sentence on the left, the retrieved law on the right. They are shown
together deliberately: the passages are gazette text with a source link and are
verified, while the sentence comes from a 77M-parameter model that, on the
held-out set, drifts section numbers to neighbouring provisions, corrupts
statutory titles and invents counterparts for repealed sections. Anyone can check
one against the other in a glance.

In [ ]:
import gradio as gr

EXAMPLES = [
    "Which BNS section replaced IPC Section 302?",
    "Is an offence under BNS Section 303 bailable?",
    "Is mischief still dealt with under IPC Section 425?",
    "Which BNS section corresponds to IPC Section 124A?",
    "Which CrPC section corresponds to BNSS Section 173?",
    "What does BNS Section 103 cover?",
]


def sources_markdown(answer):
    if answer.refused:
        return "_No sources: the question was outside what this tool answers._"
    if not answer.citations:
        return "_No passage was retrieved for this question._"
    lines = ["Retrieved from the official sources. **This is the part to trust.**", ""]
    for c in answer.citations:
        lines.append(f"- [{c['source']}]({c['source_url']})")
    return "\n".join(lines)


def ask(question):
    question = (question or "").strip()
    if not question:
        return "", "_Ask about a provision of the BNS, BNSS or BSA._"
    a = bot.ask(question)
    return a.text.strip(), sources_markdown(a)


CAVEAT = (
    "**Read the sources, not just the sentence.** The answer is written by a "
    "fine-tuned `flan-t5-small` reading the passages beside it. On the hardest "
    "held-out questions it drifts section numbers, corrupts statutory titles and "
    "invents counterparts for repealed provisions. The passages are verified; "
    "the sentence is not.")

with gr.Blocks(title="Indian criminal law after 1 July 2024",
               theme=gr.themes.Soft(primary_hue="teal")) as demo:
    gr.Markdown(
        "# Indian criminal law after 1 July 2024\n"
        "The IPC, CrPC and Evidence Act were replaced by the BNS, BNSS and BSA. "
        "Ask what a provision became, what it says, or how the First Schedule "
        "classifies it \u2014 every answer cites the gazette.")
    with gr.Row():
        box = gr.Textbox(label="Your question", scale=5, autofocus=True,
                         placeholder="Which BNS section replaced IPC Section 302?")
        btn = gr.Button("Ask", variant="primary", scale=1)
    with gr.Row():
        with gr.Column(scale=3):
            out = gr.Markdown(label="Answer")
        with gr.Column(scale=2):
            src = gr.Markdown(label="Sources")
    gr.Markdown(CAVEAT)
    gr.Examples(examples=EXAMPLES, inputs=box)
    gr.Markdown(
        f"_{DISCLAIMER}_\n\nA student capstone project, not affiliated with or "
        "endorsed by any government body. "
        "[github.com/suryanshu-g/legal-llm-bot]"
        "(https://github.com/suryanshu-g/legal-llm-bot)")

    btn.click(ask, inputs=box, outputs=[out, src])
    box.submit(ask, inputs=box, outputs=[out, src])

print("interface built")

---

## Go live

Run this and wait for the line that reads
`Running on public URL: https://….gradio.live` — **that is your link.**

In [ ]:
demo.launch(share=True, show_error=True)

---

## While it is running

* The link works for **one week**, or until this notebook stops.
* Colab disconnects an idle session after about 90 minutes; if the link dies,
  run the last cell again for a fresh one.
* Anything typed into it runs on this Colab machine, so do not share the link
  more widely than you mean to.

## Making it permanent, later

Once your Hugging Face account is more than 30 days old and its email is
verified, you can host it free on ZeroGPU: see
[`space/SETUP.md`](https://github.com/suryanshu-g/legal-llm-bot/blob/main/space/SETUP.md).

The **website** needs none of this and is already permanent, because it answers
from the concordance and the gazette rather than from the model:
<https://suryanshu-g.github.io/legal-llm-bot/>